# Homework 3

## Task 1: User Activity Dynamics Analysis

The number of active sessions is modeled by:

$$f(t) = 1000 \cdot t \cdot e^{-0.2t}$$

**Analytical derivative** using the product rule $(u \cdot v)' = u'v + uv'$, where $u = 1000t$ and $v = e^{-0.2t}$:

$$f'(t) = 1000 \cdot e^{-0.2t} + 1000t \cdot (-0.2) \cdot e^{-0.2t} = 1000 \cdot e^{-0.2t}(1 - 0.2t)$$

**Peak load** — solve $f'(t) = 0$. Since $e^{-0.2t} > 0$ for all $t$:

$$1 - 0.2t = 0 \implies t^* = 5 \text{ hours after 8:00} = \textbf{13:00}$$

In [5]:
import numpy as np

from scipy.optimize import approx_fprime
from scipy.integrate import solve_ivp
from scipy.integrate import quad
from scipy.optimize import approx_fprime

def f(t):
    return 1000 * t * np.exp(-0.2 * t)

def f_prime_analytical(t):
    return 1000 * np.exp(-0.2 * t) * (1 - 0.2 * t)

t_star = 5
hours, minutes = divmod(8 * 60 + t_star * 60, 60)
print(f"Peak load at t* = {t_star}h → {int(hours)}:{int(minutes):02d}")
print(f"Max sessions: {f(t_star):.2f}\n")

# numerical vs analytical derivative at t = 2, 6, 10
for t in [2, 6, 10]:
    numerical = approx_fprime([t], lambda x: f(x[0]), epsilon=1e-6)[0]
    analytical = f_prime_analytical(t)
    hour_label = 8 + t
    print(f"{hour_label}:00 — numerical: {numerical:.4f}, analytical: {analytical:.4f}, diff: {abs(numerical - analytical):.6f}")

print("\nBusiness interpretation:")
print("10:00 (t=2): f'(t) > 0 — sessions are growing, IT should prepare for increasing load.")
print("14:00 (t=6): f'(t) < 0 — sessions are declining, load is decreasing past its peak.")
print("18:00 (t=10): f'(t) < 0 — load is falling, resources can be scaled down.")
print(f"Recommendation: maximize server capacity around 13:00 (t* = 5h).")

Peak load at t* = 5h → 13:00
Max sessions: 1839.40

10:00 — numerical: 402.1919, analytical: 402.1920, diff: 0.000107
14:00 — numerical: -60.2389, analytical: -60.2388, diff: 0.000024
18:00 — numerical: -135.3353, analytical: -135.3353, diff: 0.000001

Business interpretation:
10:00 (t=2): f'(t) > 0 — sessions are growing, IT should prepare for increasing load.
14:00 (t=6): f'(t) < 0 — sessions are declining, load is decreasing past its peak.
18:00 (t=10): f'(t) < 0 — load is falling, resources can be scaled down.
Recommendation: maximize server capacity around 13:00 (t* = 5h).


## Task 2: Learning Process Simulation

The learning dynamic follows the ODE:

$$\frac{dK}{dt} = r(M - K), \quad M = 100,\ r = 0.15,\ K(0) = 10$$

In [6]:
M, r = 100, 0.15

def learning_rate(t, K):
    return [r * (M - K[0])]

# solve for K(0) = 10 over 30 days
sol = solve_ivp(learning_rate, [0, 30], [10], dense_output=True)
t_eval = np.linspace(0, 30, 300)
K_eval = sol.sol(t_eval)[0]

print("K(t) solved numerically over 30 days.\n")
for day in [5, 10, 15, 20, 30]:
    k_val = sol.sol(day)[0]
    print(f"Day {day:>2}: K = {k_val:.2f}%")

# compare time to reach 90% for three initial conditions
print("\nTime to reach 90% knowledge:")
for K0 in [5, 10, 20]:
    sol_k = solve_ivp(learning_rate, [0, 60], [K0], dense_output=True)
    t_fine = np.linspace(0, 60, 10000)
    K_fine = sol_k.sol(t_fine)[0]
    idx = np.argmax(K_fine >= 90)
    if idx > 0:
        print(f"K(0) = {K0}%: reaches 90% at day ≈ {t_fine[idx]:.1f}")
    else:
        print(f"K(0) = {K0}%: does not reach 90% within 60 days")

print("\nConclusion: students with higher initial knowledge reach 90% significantly faster,")
print("confirming that prior preparation has a strong compounding effect on learning speed.")

K(t) solved numerically over 30 days.

Day  5: K = 57.49%
Day 10: K = 79.92%
Day 15: K = 90.52%
Day 20: K = 95.52%
Day 30: K = 98.99%

Time to reach 90% knowledge:
K(0) = 5%: reaches 90% at day ≈ 15.0
K(0) = 10%: reaches 90% at day ≈ 14.6
K(0) = 20%: reaches 90% at day ≈ 13.9

Conclusion: students with higher initial knowledge reach 90% significantly faster,
confirming that prior preparation has a strong compounding effect on learning speed.


## Task 3: Advertising Campaign Efficiency

Registration intensity: $f(t) = 500 \cdot e^{-0.3t}$

**Analytical integral over first 7 days** using Newton–Leibniz formula:

$$\int_0^7 500 \cdot e^{-0.3t}\,dt = 500 \cdot \left[-\frac{1}{0.3}e^{-0.3t}\right]_0^7 = \frac{500}{0.3}\left(1 - e^{-2.1}\right) \approx \frac{5000}{3}(1 - e^{-2.1})$$

$$\approx 1666.67 \cdot (1 - 0.1225) \approx \mathbf{1462.6 \text{ registrations}}$$

**Theoretical maximum** (improper integral):

$$\int_0^{\infty} 500 \cdot e^{-0.3t}\,dt = \frac{500}{0.3} = \frac{5000}{3} \approx \mathbf{1666.67}$$

since $e^{-0.3t} \to 0$ as $t \to \infty$.

In [7]:
def f(t):
    return 500 * np.exp(-0.3 * t)

# analytical result
analytical_7 = (500 / 0.3) * (1 - np.exp(-0.3 * 7))
print(f"Analytical integral (0 to 7): {analytical_7:.4f} registrations")

# numerical verification
numerical_7, _ = quad(f, 0, 7)
print(f"Numerical integral  (0 to 7): {numerical_7:.4f} registrations")
print(f"Difference: {abs(analytical_7 - numerical_7):.6f}")

# theoretical maximum (improper integral)
max_theoretical, _ = quad(f, 0, np.inf)
analytical_max = 500 / 0.3
print(f"\nTheoretical maximum (0 to ∞): {analytical_max:.4f}")
print(f"Numerical (0 to ∞):           {max_theoretical:.4f}")

# first-week efficiency
efficiency = (analytical_7 / analytical_max) * 100
print(f"\nFirst-week efficiency: {efficiency:.2f}% of total theoretical registrations")
print("The first 7 days capture roughly 87.75% of all potential registrations.")

Analytical integral (0 to 7): 1462.5726 registrations
Numerical integral  (0 to 7): 1462.5726 registrations
Difference: 0.000000

Theoretical maximum (0 to ∞): 1666.6667
Numerical (0 to ∞):           1666.6667

First-week efficiency: 87.75% of total theoretical registrations
The first 7 days capture roughly 87.75% of all potential registrations.


## Task 4: Two-Variable Function Analysis

$$f(x,y) = 0.5x^2 + 0.3y^2 + 0.2xy + 10x + 5y$$

**Partial derivatives:**

$$\frac{\partial f}{\partial x} = x + 0.2y + 10$$

$$\frac{\partial f}{\partial y} = 0.6y + 0.2x + 5$$

**At point $(10,\ 20)$:**

$$\frac{\partial f}{\partial x}\bigg|_{(10,20)} = 10 + 4 + 10 = 24, \qquad \frac{\partial f}{\partial y}\bigg|_{(10,20)} = 12 + 2 + 5 = 19$$

**Linear approximation** for $\Delta x = 0.5$, $\Delta y = -0.3$:

$$\Delta f \approx 24 \cdot 0.5 + 19 \cdot (-0.3) = 12 - 5.7 = 6.3$$

In [8]:
def f(xy):
    x, y = xy
    return 0.5*x**2 + 0.3*y**2 + 0.2*x*y + 10*x + 5*y

point = np.array([10.0, 20.0])

# numerical gradient
grad_numerical = approx_fprime(point, f, epsilon=1e-6)
print(f"Numerical gradient at (10, 20): ∂f/∂x = {grad_numerical[0]:.6f}, ∂f/∂y = {grad_numerical[1]:.6f}")

# analytical gradient
df_dx = point[0] + 0.2*point[1] + 10
df_dy = 0.6*point[1] + 0.2*point[0] + 5
print(f"Analytical gradient at (10, 20): ∂f/∂x = {df_dx:.6f}, ∂f/∂y = {df_dy:.6f}")
print(f"Difference: ∂f/∂x: {abs(df_dx - grad_numerical[0]):.8f}, ∂f/∂y: {abs(df_dy - grad_numerical[1]):.8f}")

# linear approximation
dx, dy = 0.5, -0.3
delta_f_approx = df_dx * dx + df_dy * dy
print(f"\nLinear approximation Δf ≈ {delta_f_approx:.4f}")

# exact change
exact_change = f([10.5, 19.7]) - f([10.0, 20.0])
print(f"Exact change f(10.5, 19.7) - f(10, 20) = {exact_change:.4f}")
print(f"Approximation error: {abs(delta_f_approx - exact_change):.6f}")

Numerical gradient at (10, 20): ∂f/∂x = 24.000000, ∂f/∂y = 19.000000
Analytical gradient at (10, 20): ∂f/∂x = 24.000000, ∂f/∂y = 19.000000
Difference: ∂f/∂x: 0.00000047, ∂f/∂y: 0.00000027

Linear approximation Δf ≈ 6.3000
Exact change f(10.5, 19.7) - f(10, 20) = 6.4220
Approximation error: 0.122000
